In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step13.0
# Molecular subtyping project folders
# ==========================================================

import os

step13_dir = _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping')

input_dir = os.path.join(step13_dir, "Input")
results_dir = os.path.join(step13_dir, "Results")
figures_dir = os.path.join(step13_dir, "Figures")
consensus_dir = os.path.join(step13_dir, "Consensus")

for folder in [
    step13_dir,
    input_dir,
    results_dir,
    figures_dir,
    consensus_dir
]:
    os.makedirs(folder, exist_ok=True)

print(step13_dir)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step13.1
# Load required data
# ==========================================================

import pandas as pd
import numpy as np


# 1. WGCNA module assignment
module_assignment = pd.read_csv(
    _hcc_path('LIHC_RNA_Step12_WGCNA/Results/LIHC_WGCNA_Module_Assignment.csv')
)


# 2. Strict Hub table
Hub_assigned = pd.read_csv(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Input/LIHC_Strict_Assigned_Hub_Table.csv')
)


# 3. Expression matrix
# gene × sample
expression_all = pd.read_csv(
    _hcc_path('LIHC_RNA_Step11_Immune/Input/LIHC_Expression_365_log2CPM.tsv'),
    sep="\t",
    index_col=0
)


# 4. Final RiskScore + survival
risk_df = pd.read_csv(
    _hcc_path('LIHC_RNA_Step8_Final_Model/Results/Final_Top10_RiskScore.csv'),
    index_col=0
)


print("module_assignment:", module_assignment.shape)
print("Hub_assigned:", Hub_assigned.shape)
print("expression_all:", expression_all.shape)
print("risk_df:", risk_df.shape)

In [ ]:
# ==========================================================
# Step13.2
# Key WGCNA modules
# ==========================================================

key_modules = [
    "blue",
    "red",
    "brown"
]

Hub_key = Hub_assigned[
    Hub_assigned["Module"].isin(
        key_modules
    )
].copy()


print(
    Hub_key["Module"].value_counts()
)

In [ ]:
# ==========================================================
# Strategy A
# Final strict 18 Hub genes
# ==========================================================

Strategy_A_Hub18 = [

    # Blue
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",

    # Red
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",

    # Brown
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]


print(len(Strategy_A_Hub18))

In [ ]:
# ==========================================================
# Balanced Top-N strategy function
# ==========================================================

def get_top_genes_each_module(
        hub_df,
        n_each
):

    genes = []

    for module in key_modules:

        temp = (
            hub_df[
                hub_df["Module"] == module
            ]
            .sort_values(
                "Hub_score",
                ascending=False
            )
            .head(n_each)
        )

        genes.extend(
            temp["Gene"].tolist()
        )

    return genes

In [ ]:
Strategy_B_Top10 = get_top_genes_each_module(
    Hub_key,
    10
)

Strategy_C_Top20 = get_top_genes_each_module(
    Hub_key,
    20
)

Strategy_D_Top30 = get_top_genes_each_module(
    Hub_key,
    30
)


print("Hub18:", len(Strategy_A_Hub18))
print("Top10:", len(Strategy_B_Top10))
print("Top20:", len(Strategy_C_Top20))
print("Top30:", len(Strategy_D_Top30))

In [ ]:
# ==========================================================
# Strategy E
# Unified MM + GS threshold
# ==========================================================

Strategy_E_MMGS = (
    Hub_key[
        (Hub_key["MM"].abs() >= 0.80)
        &
        (Hub_key["GS"].abs() >= 0.20)
    ]
    ["Gene"]
    .tolist()
)


print(
    "MM>=0.80 & |GS|>=0.20:",
    len(Strategy_E_MMGS)
)

In [ ]:
# ==========================================================
# Strategy F
# All genes in Blue + Red + Brown
# ==========================================================

Strategy_F_AllModules = (
    Hub_key["Gene"]
    .drop_duplicates()
    .tolist()
)


print(
    "All key-module genes:",
    len(Strategy_F_AllModules)
)

In [ ]:
# ==========================================================
# Collect all strategies
# ==========================================================

gene_strategies = {

    "A_Hub18":
        Strategy_A_Hub18,

    "B_Top10_each":
        Strategy_B_Top10,

    "C_Top20_each":
        Strategy_C_Top20,

    "D_Top30_each":
        Strategy_D_Top30,

    "E_MM80_GS20":
        Strategy_E_MMGS,

    "F_AllModules":
        Strategy_F_AllModules
}

In [ ]:
# ==========================================================
# Strategy summary
# ==========================================================

strategy_summary = []


for strategy_name, gene_list in gene_strategies.items():

    temp = module_assignment[
        module_assignment["Gene"].isin(
            gene_list
        )
    ]


    counts = temp["Module"].value_counts()


    strategy_summary.append({

        "Strategy":
            strategy_name,

        "N_genes":
            len(gene_list),

        "Blue":
            counts.get("blue", 0),

        "Red":
            counts.get("red", 0),

        "Brown":
            counts.get("brown", 0)
    })


strategy_summary = pd.DataFrame(
    strategy_summary
)


strategy_summary

In [ ]:
# ==========================================================
# Save strategies
# ==========================================================

for strategy_name, gene_list in gene_strategies.items():

    pd.DataFrame(
        {
            "Gene": gene_list
        }
    ).to_csv(
        os.path.join(
            input_dir,
            f"{strategy_name}_genes.csv"
        ),
        index=False
    )


strategy_summary.to_csv(
    os.path.join(
        results_dir,
        "Molecular_Subtyping_Gene_Strategy_Summary.csv"
    ),
    index=False
)


print("Saved.")

In [ ]:
# ==========================================================
# Step13.10
# Consensus clustering function
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


def consensus_clustering(
    expr_gene_sample,
    k,
    n_resampling=200,
    sample_fraction=0.80,
    random_state=42
):

    """
    expr_gene_sample:
        gene x sample expression matrix

    k:
        number of clusters

    n_resampling:
        number of resampling iterations

    sample_fraction:
        fraction of samples used in each iteration
    """

    # ----------------------------------
    # gene x sample -> sample x gene
    # ----------------------------------

    X_df = expr_gene_sample.T.copy()

    samples = X_df.index.to_numpy()

    X = X_df.values


    # ----------------------------------
    # Standardize genes
    # ----------------------------------

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)


    n_samples = X_scaled.shape[0]


    # ----------------------------------
    # Matrices
    # ----------------------------------

    consensus_count = np.zeros(
        (n_samples, n_samples),
        dtype=float
    )

    co_sample_count = np.zeros(
        (n_samples, n_samples),
        dtype=float
    )


    rng = np.random.default_rng(
        random_state
    )


    n_sub = int(
        np.floor(
            sample_fraction * n_samples
        )
    )


    # ----------------------------------
    # Resampling
    # ----------------------------------

    for i in range(n_resampling):

        selected_idx = rng.choice(
            n_samples,
            size=n_sub,
            replace=False
        )

        X_sub = X_scaled[
            selected_idx
        ]


        km = KMeans(
            n_clusters=k,
            n_init=20,
            random_state=random_state + i
        )


        labels_sub = km.fit_predict(
            X_sub
        )


        # samples that were together in this resampling
        co_sample_count[
            np.ix_(
                selected_idx,
                selected_idx
            )
        ] += 1


        for cluster_id in range(k):

            cluster_members = selected_idx[
                labels_sub == cluster_id
            ]


            consensus_count[
                np.ix_(
                    cluster_members,
                    cluster_members
                )
            ] += 1


    # ----------------------------------
    # Consensus matrix
    # ----------------------------------

    consensus_matrix = np.divide(
        consensus_count,
        co_sample_count,
        out=np.zeros_like(
            consensus_count
        ),
        where=co_sample_count != 0
    )


    np.fill_diagonal(
        consensus_matrix,
        1
    )


    # ----------------------------------
    # Final clustering
    # based on full dataset
    # ----------------------------------

    final_km = KMeans(
        n_clusters=k,
        n_init=100,
        random_state=random_state
    )


    final_labels = final_km.fit_predict(
        X_scaled
    )


    # cluster label from 1
    final_labels = (
        final_labels + 1
    )


    # ----------------------------------
    # Silhouette
    # ----------------------------------

    silhouette = silhouette_score(
        X_scaled,
        final_labels
    )


    cluster_series = pd.Series(
        final_labels,
        index=samples,
        name="Cluster"
    )


    consensus_df = pd.DataFrame(
        consensus_matrix,
        index=samples,
        columns=samples
    )


    return {
        "clusters": cluster_series,
        "consensus_matrix": consensus_df,
        "silhouette": silhouette
    }

In [ ]:
# ==========================================================
# Step13.11
# Test Strategy A: 18 Hub genes
# ==========================================================

genes_A = gene_strategies[
    "A_Hub18"
]


expr_A = expression_all.loc[
    genes_A
].copy()


print(
    expr_A.shape
)

In [ ]:
# K = 2

test_A_K2 = consensus_clustering(
    expr_A,
    k=2,
    n_resampling=200,
    sample_fraction=0.80,
    random_state=42
)


print(
    "Silhouette:",
    test_A_K2["silhouette"]
)


print(
    test_A_K2[
        "clusters"
    ].value_counts()
)

In [ ]:
# ==========================================================
# Step13.12
# Strategy A: test K = 2~6
# ==========================================================

A_results = []


for k in range(2, 7):

    result = consensus_clustering(
        expr_A,
        k=k,
        n_resampling=200,
        sample_fraction=0.80,
        random_state=42
    )

    cluster_counts = (
        result["clusters"]
        .value_counts()
        .sort_index()
    )

    A_results.append({
        "Strategy": "A_Hub18",
        "K": k,
        "Silhouette": result["silhouette"],
        "Min_cluster_size": cluster_counts.min(),
        "Max_cluster_size": cluster_counts.max(),
        "Cluster_sizes": cluster_counts.to_dict()
    })

    print(
        f"K={k} | "
        f"Silhouette={result['silhouette']:.4f} | "
        f"Sizes={cluster_counts.to_dict()}"
    )

In [ ]:
A_summary = pd.DataFrame(A_results)

A_summary

In [ ]:
A_summary.to_csv(
    os.path.join(
        results_dir,
        "Strategy_A_Hub18_K2_K6_summary.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.15.1
# Run and store full consensus results
# Strategy A: Hub18
# K = 2~6
# ==========================================================

A_consensus_results = {}


for k in range(2, 7):

    print(f"Running K={k} ...")

    result = consensus_clustering(
        expr_A,
        k=k,
        n_resampling=200,
        sample_fraction=0.80,
        random_state=42
    )

    A_consensus_results[k] = result


print("Finished.")

In [ ]:
# ==========================================================
# Step13.15.2
# PAC calculation
# Lower PAC = more stable clustering
# ==========================================================

def calculate_PAC(
        consensus_matrix,
        lower=0.10,
        upper=0.90
):

    C = consensus_matrix.values.copy()


    # 只取上三角，排除对角线和重复样本对
    upper_triangle = C[
        np.triu_indices_from(
            C,
            k=1
        )
    ]


    # ambiguous consensus proportion
    PAC = np.mean(
        (upper_triangle > lower)
        &
        (upper_triangle < upper)
    )


    return PAC

In [ ]:
# ==========================================================
# Calculate PAC for K = 2~6
# ==========================================================

PAC_results = []


for k in range(2, 7):

    consensus_matrix = (
        A_consensus_results[k][
            "consensus_matrix"
        ]
    )


    pac = calculate_PAC(
        consensus_matrix,
        lower=0.10,
        upper=0.90
    )


    silhouette = (
        A_consensus_results[k][
            "silhouette"
        ]
    )


    cluster_counts = (
        A_consensus_results[k][
            "clusters"
        ]
        .value_counts()
        .sort_index()
    )


    PAC_results.append({

        "Strategy":
            "A_Hub18",

        "K":
            k,

        "PAC":
            pac,

        "Silhouette":
            silhouette,

        "Min_cluster_size":
            cluster_counts.min(),

        "Max_cluster_size":
            cluster_counts.max(),

        "Cluster_sizes":
            cluster_counts.to_dict()

    })


A_PAC_summary = pd.DataFrame(
    PAC_results
)


A_PAC_summary

In [ ]:
# ==========================================================
# Rank K by PAC
# ==========================================================

A_PAC_ranked = (
    A_PAC_summary
    .sort_values(
        [
            "PAC",
            "Silhouette"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


A_PAC_ranked

In [ ]:
A_PAC_summary.to_csv(
    os.path.join(
        results_dir,
        "Strategy_A_Hub18_PAC_K2_K6.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.16
# Consensus Matrix Heatmap
# Strategy A: Hub18
# Best candidate K = 2
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ----------------------------------------------------------
# 1. 获取 K=2 结果
# ----------------------------------------------------------

k = 2

consensus_df = (
    A_consensus_results[k][
        "consensus_matrix"
    ]
    .copy()
)

cluster_labels = (
    A_consensus_results[k][
        "clusters"
    ]
    .copy()
)


print("Consensus matrix:", consensus_df.shape)
print(cluster_labels.value_counts().sort_index())

In [ ]:
# ==========================================================
# 2. Sort samples according to cluster
# ==========================================================

sample_order = (
    cluster_labels
    .sort_values()
    .index
)


consensus_sorted = consensus_df.loc[
    sample_order,
    sample_order
]


cluster_sorted = cluster_labels.loc[
    sample_order
]


print(consensus_sorted.shape)

In [ ]:
# ==========================================================
# 3. Cluster boundary
# ==========================================================

cluster_sizes = (
    cluster_sorted
    .value_counts()
    .sort_index()
)


print(cluster_sizes)


boundaries = np.cumsum(
    cluster_sizes.values
)


print(
    "Cluster boundaries:",
    boundaries
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# 4. Consensus matrix heatmap
# ==========================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)


sns.heatmap(
    consensus_sorted,

    cmap="Blues",

    vmin=0,
    vmax=1,

    square=True,

    xticklabels=False,
    yticklabels=False,

    cbar_kws={
        "label": "Consensus index"
    },

    ax=ax
)


# ----------------------------------------------------------
# Add cluster boundaries
# ----------------------------------------------------------

for boundary in boundaries[:-1]:

    ax.axhline(
        boundary,
        linewidth=1.5
    )

    ax.axvline(
        boundary,
        linewidth=1.5
    )


ax.set_title(
    "Consensus Matrix - A_Hub18 (K=2)",
    fontsize=14,
    pad=12
)


ax.set_xlabel(
    "Samples"
)

ax.set_ylabel(
    "Samples"
)


fig.tight_layout()


# ----------------------------------------------------------
# Save BEFORE show
# ----------------------------------------------------------

fig.savefig(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Figures/A_Hub18_K2_Consensus_Matrix.pdf'),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Figures/A_Hub18_K2_Consensus_Matrix.png'),
    dpi=600,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)

In [ ]:
# ==========================================================
# Step13.17
# Batch consensus clustering
# All strategies, K = 2~6
# ==========================================================

all_consensus_results = {}

summary_rows = []


for strategy_name, gene_list in gene_strategies.items():

    print(
        "\n",
        "=" * 60
    )

    print(
        "Running strategy:",
        strategy_name,
        "| genes:",
        len(gene_list)
    )

    print(
        "=" * 60
    )


    # ----------------------------------
    # expression matrix
    # gene × sample
    # ----------------------------------

    expr_strategy = expression_all.loc[
        gene_list
    ].copy()


    all_consensus_results[
        strategy_name
    ] = {}


    # ----------------------------------
    # K = 2~6
    # ----------------------------------

    for k in range(2, 7):

        print(
            f"  K={k} ..."
        )


        result = consensus_clustering(
            expr_strategy,
            k=k,
            n_resampling=200,
            sample_fraction=0.80,
            random_state=42
        )


        all_consensus_results[
            strategy_name
        ][k] = result


        # ----------------------------------
        # PAC
        # ----------------------------------

        pac = calculate_PAC(
            result[
                "consensus_matrix"
            ],
            lower=0.10,
            upper=0.90
        )


        # ----------------------------------
        # Cluster sizes
        # ----------------------------------

        cluster_counts = (
            result[
                "clusters"
            ]
            .value_counts()
            .sort_index()
        )


        summary_rows.append({

            "Strategy":
                strategy_name,

            "N_genes":
                len(gene_list),

            "K":
                k,

            "PAC":
                pac,

            "Silhouette":
                result[
                    "silhouette"
                ],

            "Min_cluster_size":
                cluster_counts.min(),

            "Max_cluster_size":
                cluster_counts.max(),

            "Cluster_sizes":
                cluster_counts.to_dict()

        })


print("\nAll strategies finished.")

In [ ]:
# ==========================================================
# Step13.18
# Summary table
# ==========================================================

Consensus_summary = pd.DataFrame(
    summary_rows
)


Consensus_summary = (
    Consensus_summary
    .sort_values(
        [
            "Strategy",
            "K"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    Consensus_summary
)

In [ ]:
Consensus_summary.to_csv(
    os.path.join(
        results_dir,
        "All_Strategies_Consensus_K2_K6_Summary.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.19
# Best K candidate for each strategy
# Primary ranking: PAC low
# Secondary ranking: Silhouette high
# ==========================================================

best_K_rows = []


for strategy_name in gene_strategies.keys():

    temp = (
        Consensus_summary[
            Consensus_summary[
                "Strategy"
            ] == strategy_name
        ]
        .copy()
    )


    # 先剔除极小cluster
    # 这里暂定最小cluster >= 20
    temp_valid = temp[
        temp[
            "Min_cluster_size"
        ] >= 20
    ].copy()


    # 如果全部都不满足，再退回全部结果
    if len(temp_valid) == 0:
        temp_valid = temp.copy()


    best_row = (
        temp_valid
        .sort_values(
            [
                "PAC",
                "Silhouette"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[0]
    )


    best_K_rows.append(
        best_row
    )


Best_K_summary = pd.DataFrame(
    best_K_rows
).reset_index(
    drop=True
)


display(
    Best_K_summary[
        [
            "Strategy",
            "N_genes",
            "K",
            "PAC",
            "Silhouette",
            "Min_cluster_size",
            "Max_cluster_size",
            "Cluster_sizes"
        ]
    ]
)

In [ ]:
Best_K_summary.to_csv(
    os.path.join(
        results_dir,
        "Best_K_Per_Strategy_Preliminary.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.20.1
# Extract cluster labels for 3 candidate strategies
# ==========================================================

candidate_strategies = {
    "A_Hub18": 2,
    "D_Top30_each": 2,
    "E_MM80_GS20": 2
}


cluster_label_dict = {}


for strategy_name, k in candidate_strategies.items():

    labels = (
        all_consensus_results[
            strategy_name
        ][k][
            "clusters"
        ]
        .copy()
    )

    labels.name = strategy_name

    cluster_label_dict[
        strategy_name
    ] = labels


# 合并成一个表
cluster_labels_all = pd.concat(
    cluster_label_dict.values(),
    axis=1
)


print(cluster_labels_all.shape)

cluster_labels_all.head()

In [ ]:
# ==========================================================
# Cluster sizes
# ==========================================================

for strategy in candidate_strategies.keys():

    print(
        f"\n===== {strategy} ====="
    )

    print(
        cluster_labels_all[
            strategy
        ]
        .value_counts()
        .sort_index()
    )

In [ ]:
cluster_labels_all.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtype_Cluster_Labels.csv"
    )
)

print("Saved.")

In [ ]:
# ==========================================================
# Step13.21
# Merge cluster labels with RiskScore
# ==========================================================

cluster_risk_df = (
    cluster_labels_all
    .join(
        risk_df[
            [
                "RiskScore",
                "OS_time",
                "OS"
            ]
        ],
        how="inner"
    )
)


print(
    cluster_risk_df.shape
)

cluster_risk_df.head()

In [ ]:
# ==========================================================
# RiskScore comparison between subtype clusters
# ==========================================================

from scipy.stats import mannwhitneyu


risk_compare_results = []


for strategy in candidate_strategies.keys():

    group1 = cluster_risk_df.loc[
        cluster_risk_df[
            strategy
        ] == 1,
        "RiskScore"
    ]

    group2 = cluster_risk_df.loc[
        cluster_risk_df[
            strategy
        ] == 2,
        "RiskScore"
    ]


    stat, p = mannwhitneyu(
        group1,
        group2,
        alternative="two-sided"
    )


    risk_compare_results.append({

        "Strategy":
            strategy,

        "Cluster1_n":
            len(group1),

        "Cluster2_n":
            len(group2),

        "Cluster1_median_RiskScore":
            group1.median(),

        "Cluster2_median_RiskScore":
            group2.median(),

        "Median_difference_C1_minus_C2":
            group1.median()
            -
            group2.median(),

        "P_value":
            p
    })


RiskScore_Subtype_Comparison = pd.DataFrame(
    risk_compare_results
)


RiskScore_Subtype_Comparison

In [ ]:
RiskScore_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_RiskScore_Comparison.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.22
# Overall survival comparison
# Log-rank test
# ==========================================================

from lifelines.statistics import logrank_test


survival_compare_results = []


for strategy in candidate_strategies.keys():

    group1 = cluster_risk_df[
        cluster_risk_df[
            strategy
        ] == 1
    ]

    group2 = cluster_risk_df[
        cluster_risk_df[
            strategy
        ] == 2
    ]


    lr = logrank_test(
        group1["OS_time"],
        group2["OS_time"],
        event_observed_A=group1["OS"],
        event_observed_B=group2["OS"]
    )


    survival_compare_results.append({

        "Strategy":
            strategy,

        "Cluster1_n":
            len(group1),

        "Cluster2_n":
            len(group2),

        "Cluster1_events":
            int(group1["OS"].sum()),

        "Cluster2_events":
            int(group2["OS"].sum()),

        "LogRank_P":
            lr.p_value
    })


Subtype_OS_Comparison = pd.DataFrame(
    survival_compare_results
)


Subtype_OS_Comparison

In [ ]:
Subtype_OS_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_OS_Comparison.csv"
    ),
    index=False
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Reload module eigengene matrix
# ==========================================================

import pandas as pd

ME_matrix = pd.read_csv(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Input/LIHC_Module_Eigengenes_365.csv'),
    index_col=0
)

print(ME_matrix.shape)
print(ME_matrix.columns.tolist())

ME_matrix.head()

In [ ]:
module_cols = [
    "ME_blue",
    "ME_red",
    "ME_brown"
]

cluster_ME_df = (
    cluster_labels_all
    .join(
        ME_matrix[
            module_cols
        ],
        how="inner"
    )
)

print(cluster_ME_df.shape)
cluster_ME_df.head()

In [ ]:
# ==========================================================
# Step13.23.2
# Compare module eigengenes between Cluster 1 and Cluster 2
# ==========================================================

from scipy.stats import mannwhitneyu
import pandas as pd


ME_compare_results = []


for strategy in candidate_strategies.keys():

    for module in module_cols:

        group1 = cluster_ME_df.loc[
            cluster_ME_df[strategy] == 1,
            module
        ]

        group2 = cluster_ME_df.loc[
            cluster_ME_df[strategy] == 2,
            module
        ]


        stat, p = mannwhitneyu(
            group1,
            group2,
            alternative="two-sided"
        )


        ME_compare_results.append({

            "Strategy": strategy,

            "Module": module,

            "Cluster1_n": len(group1),

            "Cluster2_n": len(group2),

            "Cluster1_median": group1.median(),

            "Cluster2_median": group2.median(),

            "Median_difference_C1_minus_C2":
                group1.median() - group2.median(),

            "P_value": p
        })


Module_Subtype_Comparison = pd.DataFrame(
    ME_compare_results
)


Module_Subtype_Comparison

In [ ]:
# ==========================================================
# Step13.23.3
# BH-FDR correction
# ==========================================================

from statsmodels.stats.multitest import multipletests


Module_Subtype_Comparison["FDR"] = multipletests(
    Module_Subtype_Comparison["P_value"],
    method="fdr_bh"
)[1]


Module_Subtype_Comparison

In [ ]:
Module_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_Module_Eigengene_Comparison.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.24.1
# Load integrated immune matrix
# ==========================================================

import os
import pandas as pd
import numpy as np

immune_file = (
    r"E:\TCGA\LIHC_RNA_Step11_Immune\Results"
    r"\LIHC_Integrated_Immune_Matrix_365.csv"
)

immune_df = pd.read_csv(
    immune_file,
    index_col=0
)

print("Immune matrix:", immune_df.shape)
print("Missing values:", immune_df.isna().sum().sum())

print("\nImmune features:")
print(immune_df.columns.tolist())

In [ ]:
# ==========================================================
# Merge subtype labels + immune features
# ==========================================================

cluster_immune_df = (
    cluster_labels_all
    .join(
        immune_df,
        how="inner"
    )
)

print(cluster_immune_df.shape)

cluster_immune_df.head()

In [ ]:
# ==========================================================
# Step13.24.3
# Immune comparison between C1 and C2
# Mann-Whitney U + effect size
# ==========================================================

from scipy.stats import mannwhitneyu

immune_features = immune_df.columns.tolist()

immune_compare_results = []


for strategy in candidate_strategies.keys():

    for feature in immune_features:

        group1 = (
            cluster_immune_df.loc[
                cluster_immune_df[strategy] == 1,
                feature
            ]
            .dropna()
        )

        group2 = (
            cluster_immune_df.loc[
                cluster_immune_df[strategy] == 2,
                feature
            ]
            .dropna()
        )


        U, p = mannwhitneyu(
            group1,
            group2,
            alternative="two-sided"
        )


        # --------------------------------------
        # Rank-biserial correlation
        # + : Cluster1 higher
        # - : Cluster2 higher
        # --------------------------------------

        effect = (
            2 * U / (len(group1) * len(group2))
            - 1
        )


        immune_compare_results.append({

            "Strategy":
                strategy,

            "Feature":
                feature,

            "Cluster1_n":
                len(group1),

            "Cluster2_n":
                len(group2),

            "Cluster1_median":
                group1.median(),

            "Cluster2_median":
                group2.median(),

            "Effect_C1_vs_C2":
                effect,

            "P_value":
                p

        })


Immune_Subtype_Comparison = pd.DataFrame(
    immune_compare_results
)

print(
    Immune_Subtype_Comparison.shape
)

Immune_Subtype_Comparison.head()

In [ ]:
# ==========================================================
# Step13.24.4
# BH-FDR correction within each strategy
# ==========================================================

from statsmodels.stats.multitest import multipletests

Immune_Subtype_Comparison["FDR"] = np.nan


for strategy in candidate_strategies.keys():

    mask = (
        Immune_Subtype_Comparison["Strategy"]
        == strategy
    )

    Immune_Subtype_Comparison.loc[
        mask,
        "FDR"
    ] = multipletests(
        Immune_Subtype_Comparison.loc[
            mask,
            "P_value"
        ],
        method="fdr_bh"
    )[1]


Immune_Subtype_Comparison["Significant"] = (
    Immune_Subtype_Comparison["FDR"] < 0.05
)

Immune_Subtype_Comparison.head()

In [ ]:
Immune_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_Immune_Comparison.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Number of significant immune features
# ==========================================================

immune_sig_count = (
    Immune_Subtype_Comparison
    .groupby("Strategy")["Significant"]
    .agg(
        Significant_features="sum",
        Total_features="count"
    )
    .reset_index()
)

immune_sig_count

In [ ]:
# ==========================================================
# Top immune differences by effect size
# ==========================================================

top_immune_results = []


for strategy in candidate_strategies.keys():

    temp = (
        Immune_Subtype_Comparison[
            (
                Immune_Subtype_Comparison["Strategy"]
                == strategy
            )
            &
            (
                Immune_Subtype_Comparison["FDR"]
                < 0.05
            )
        ]
        .copy()
    )

    temp["Abs_effect"] = (
        temp["Effect_C1_vs_C2"].abs()
    )

    temp = (
        temp
        .sort_values(
            "Abs_effect",
            ascending=False
        )
        .head(10)
    )

    top_immune_results.append(
        temp
    )


Top10_Immune_Per_Strategy = pd.concat(
    top_immune_results,
    ignore_index=True
)


display(
    Top10_Immune_Per_Strategy[
        [
            "Strategy",
            "Feature",
            "Cluster1_median",
            "Cluster2_median",
            "Effect_C1_vs_C2",
            "FDR"
        ]
    ]
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step13.25.1
# Load clinical data and inspect variables
# ==========================================================

import os
import pandas as pd
import numpy as np

clinical_file = _hcc_path('LIHC_clinical_clean.csv')

clinical_df = pd.read_csv(
    clinical_file
)

print("Clinical shape:", clinical_df.shape)

print("\nClinical columns:")
for i, col in enumerate(clinical_df.columns):
    print(i, col)

clinical_df.head()

In [ ]:
# ==========================================================
# Check ID-related columns
# ==========================================================

id_candidates = [
    col for col in clinical_df.columns
    if any(
        key in col.lower()
        for key in [
            "sample",
            "patient",
            "submitter",
            "case"
        ]
    )
]

print("Possible ID columns:")
print(id_candidates)

In [ ]:
# ==========================================================
# Check subtype sample IDs
# ==========================================================

print("Cluster labels shape:", cluster_labels_all.shape)

print("\nFirst 10 subtype IDs:")
print(
    cluster_labels_all.index[:10].tolist()
)

In [ ]:
# ==========================================================
# Step13.25.2
# Merge subtype labels with clinical variables
# ==========================================================

clinical_tumor = (
    clinical_df[
        clinical_df["Tumor_Normal"] == "Tumor"
    ]
    .copy()
)

clinical_tumor = (
    clinical_tumor
    .drop_duplicates(
        subset="Sample_ID"
    )
    .set_index("Sample_ID")
)


clinical_vars = [
    "Age",
    "Gender",
    "Stage",
    "T",
    "N",
    "M",
    "Grade",
    "Vital_Status"
]


clinical_subtype_df = (
    cluster_labels_all
    .join(
        clinical_tumor[
            clinical_vars
        ],
        how="left"
    )
)


print(
    "Merged shape:",
    clinical_subtype_df.shape
)

print("\nMissing values:")
print(
    clinical_subtype_df[
        clinical_vars
    ].isna().sum()
)

clinical_subtype_df.head()

In [ ]:
# ==========================================================
# Step13.25.3
# Clean clinical categories
# ==========================================================

clinical_analysis_df = clinical_subtype_df.copy()


# ----------------------------------------------------------
# 1. Age
# 连续变量保留
# ----------------------------------------------------------

clinical_analysis_df["Age"] = pd.to_numeric(
    clinical_analysis_df["Age"],
    errors="coerce"
)


# ----------------------------------------------------------
# 2. Gender
# ----------------------------------------------------------

clinical_analysis_df["Gender_group"] = (
    clinical_analysis_df["Gender"]
    .str.lower()
    .map({
        "male": "Male",
        "female": "Female"
    })
)


# ----------------------------------------------------------
# 3. Stage
# 合并成 Stage I / II / III-IV
# ----------------------------------------------------------

def simplify_stage(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper()

    if "STAGE I" in x and "STAGE II" not in x and "STAGE III" not in x and "STAGE IV" not in x:
        return "Stage I"

    elif "STAGE II" in x and "STAGE III" not in x:
        return "Stage II"

    elif "STAGE III" in x or "STAGE IV" in x:
        return "Stage III-IV"

    else:
        return np.nan


clinical_analysis_df["Stage_group"] = (
    clinical_analysis_df["Stage"]
    .apply(simplify_stage)
)


# ----------------------------------------------------------
# 4. T stage
# 合并 T1 / T2 / T3-4
# TX -> missing
# ----------------------------------------------------------

def simplify_T(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper()

    if x.startswith("T1"):
        return "T1"

    elif x.startswith("T2"):
        return "T2"

    elif x.startswith("T3") or x.startswith("T4"):
        return "T3-4"

    else:
        return np.nan


clinical_analysis_df["T_group"] = (
    clinical_analysis_df["T"]
    .apply(simplify_T)
)


# ----------------------------------------------------------
# 5. N stage
# NX -> missing
# ----------------------------------------------------------

def simplify_N(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper()

    if x == "N0":
        return "N0"

    elif x.startswith("N1"):
        return "N1"

    else:
        return np.nan


clinical_analysis_df["N_group"] = (
    clinical_analysis_df["N"]
    .apply(simplify_N)
)


# ----------------------------------------------------------
# 6. M stage
# MX -> missing
# ----------------------------------------------------------

def simplify_M(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper()

    if x == "M0":
        return "M0"

    elif x.startswith("M1"):
        return "M1"

    else:
        return np.nan


clinical_analysis_df["M_group"] = (
    clinical_analysis_df["M"]
    .apply(simplify_M)
)


# ----------------------------------------------------------
# Check categories
# ----------------------------------------------------------

for col in [
    "Gender_group",
    "Stage_group",
    "T_group",
    "N_group",
    "M_group",
    "Grade"
]:

    print(
        f"\n===== {col} ====="
    )

    print(
        clinical_analysis_df[col]
        .value_counts(
            dropna=False
        )
    )

In [ ]:
# ==========================================================
# Age comparison
# ==========================================================

from scipy.stats import mannwhitneyu

age_results = []


for strategy in candidate_strategies.keys():

    g1 = (
        clinical_analysis_df.loc[
            clinical_analysis_df[strategy] == 1,
            "Age"
        ]
        .dropna()
    )

    g2 = (
        clinical_analysis_df.loc[
            clinical_analysis_df[strategy] == 2,
            "Age"
        ]
        .dropna()
    )


    U, p = mannwhitneyu(
        g1,
        g2,
        alternative="two-sided"
    )


    age_results.append({

        "Strategy":
            strategy,

        "Cluster1_n":
            len(g1),

        "Cluster2_n":
            len(g2),

        "Cluster1_median_age":
            g1.median(),

        "Cluster2_median_age":
            g2.median(),

        "P_value":
            p

    })


Age_Subtype_Comparison = pd.DataFrame(
    age_results
)

Age_Subtype_Comparison

In [ ]:
# ==========================================================
# Categorical clinical characteristics
# Chi-square test
# ==========================================================

from scipy.stats import chi2_contingency


categorical_vars = [
    "Gender_group",
    "Stage_group",
    "T_group",
    "N_group",
    "M_group"
]


clinical_compare_results = []


for strategy in candidate_strategies.keys():

    for var in categorical_vars:

        temp = (
            clinical_analysis_df[
                [strategy, var]
            ]
            .dropna()
        )


        contingency = pd.crosstab(
            temp[strategy],
            temp[var]
        )


        # 至少2×2才能检验
        if (
            contingency.shape[0] >= 2
            and
            contingency.shape[1] >= 2
        ):

            chi2, p, dof, expected = (
                chi2_contingency(
                    contingency
                )
            )

            min_expected = (
                expected.min()
            )

        else:

            chi2 = np.nan
            p = np.nan
            dof = np.nan
            min_expected = np.nan


        clinical_compare_results.append({

            "Strategy":
                strategy,

            "Variable":
                var,

            "N_available":
                len(temp),

            "Chi2":
                chi2,

            "df":
                dof,

            "Min_expected_count":
                min_expected,

            "P_value":
                p

        })


Clinical_Subtype_Comparison = pd.DataFrame(
    clinical_compare_results
)


Clinical_Subtype_Comparison

In [ ]:
# ==========================================================
# BH-FDR correction
# ==========================================================

from statsmodels.stats.multitest import multipletests


Clinical_Subtype_Comparison["FDR"] = np.nan


for strategy in candidate_strategies.keys():

    mask = (
        Clinical_Subtype_Comparison[
            "Strategy"
        ] == strategy
    )

    valid = (
        mask
        &
        Clinical_Subtype_Comparison[
            "P_value"
        ].notna()
    )


    Clinical_Subtype_Comparison.loc[
        valid,
        "FDR"
    ] = multipletests(
        Clinical_Subtype_Comparison.loc[
            valid,
            "P_value"
        ],
        method="fdr_bh"
    )[1]


Clinical_Subtype_Comparison

In [ ]:
Age_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_Age_Comparison.csv"
    ),
    index=False
)


Clinical_Subtype_Comparison.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_Clinical_Comparison.csv"
    ),
    index=False
)


clinical_analysis_df.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_Clinical_Merged.csv"
    )
)

In [ ]:
# ==========================================================
# Step13.25.7
# Clinical distribution + Fisher exact test for N/M
# ==========================================================

from scipy.stats import fisher_exact
import pandas as pd
import numpy as np


# ----------------------------------------------------------
# 1. Stage / T distribution
# ----------------------------------------------------------

for strategy in candidate_strategies.keys():

    print("\n" + "=" * 60)
    print(strategy)
    print("=" * 60)

    for var in ["Stage_group", "T_group"]:

        print(f"\n--- {var}: counts ---")

        tab = pd.crosstab(
            clinical_analysis_df[strategy],
            clinical_analysis_df[var]
        )

        display(tab)

        print(f"\n--- {var}: row percentages ---")

        tab_pct = pd.crosstab(
            clinical_analysis_df[strategy],
            clinical_analysis_df[var],
            normalize="index"
        ) * 100

        display(
            tab_pct.round(1)
        )

In [ ]:
# ==========================================================
# Fisher exact test for sparse N / M tables
# ==========================================================

fisher_results = []


for strategy in candidate_strategies.keys():

    for var in [
        "N_group",
        "M_group"
    ]:

        temp = (
            clinical_analysis_df[
                [strategy, var]
            ]
            .dropna()
        )

        table = pd.crosstab(
            temp[strategy],
            temp[var]
        )


        if table.shape == (2, 2):

            odds_ratio, p = fisher_exact(
                table.values,
                alternative="two-sided"
            )

        else:

            odds_ratio = np.nan
            p = np.nan


        fisher_results.append({

            "Strategy": strategy,
            "Variable": var,
            "N_available": len(temp),
            "Odds_ratio": odds_ratio,
            "Fisher_P": p

        })


Clinical_Fisher_NM = pd.DataFrame(
    fisher_results
)

Clinical_Fisher_NM

In [ ]:
Clinical_Fisher_NM.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_NM_Fisher_Test.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step13.25.8
# Fisher exact test for N / M
# ==========================================================

from scipy.stats import fisher_exact
import pandas as pd
import numpy as np

fisher_results = []

for strategy in candidate_strategies.keys():

    for var in ["N_group", "M_group"]:

        temp = clinical_analysis_df[
            [strategy, var]
        ].dropna()

        table = pd.crosstab(
            temp[strategy],
            temp[var]
        )

        print("\n", strategy, var)
        display(table)

        if table.shape == (2, 2):

            odds_ratio, p = fisher_exact(
                table.values,
                alternative="two-sided"
            )

        else:

            odds_ratio = np.nan
            p = np.nan

        fisher_results.append({
            "Strategy": strategy,
            "Variable": var,
            "N_available": len(temp),
            "Odds_ratio": odds_ratio,
            "Fisher_P": p
        })


Clinical_Fisher_NM = pd.DataFrame(
    fisher_results
)

Clinical_Fisher_NM

In [ ]:
Clinical_Fisher_NM.to_csv(
    os.path.join(
        results_dir,
        "Candidate_Subtypes_NM_Fisher_Test.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Save all 3 retained subtype solutions
# ==========================================================

Subtype_Solutions = pd.DataFrame(
    index=cluster_labels_all.index
)

Subtype_Solutions["A_Hub18_K2"] = (
    cluster_labels_all["A_Hub18"]
)

Subtype_Solutions["D_Top30_each_K2"] = (
    cluster_labels_all["D_Top30_each"]
)

Subtype_Solutions["E_MM80_GS20_K2"] = (
    cluster_labels_all["E_MM80_GS20"]
)


# ----------------------------------------------------------
# Add biological names
# 目前三个方案方向一致：
# Cluster 1 = Metabolic
# Cluster 2 = Proliferative
# ----------------------------------------------------------

for col in [
    "A_Hub18_K2",
    "D_Top30_each_K2",
    "E_MM80_GS20_K2"
]:

    Subtype_Solutions[
        col + "_Name"
    ] = Subtype_Solutions[col].map({
        1: "Metabolic",
        2: "Proliferative"
    })


print(Subtype_Solutions.shape)

Subtype_Solutions.head()

In [ ]:
Subtype_Solutions.to_csv(
    os.path.join(
        results_dir,
        "LIHC_Retained_3_Molecular_Subtype_Solutions.csv"
    )
)

print("Saved.")

In [ ]:
# ==========================================================
# Step13.26
# Fix final molecular subtype
# Primary strategy: D_Top30_each, K=2
# ==========================================================

final_subtype = (
    all_consensus_results[
        "D_Top30_each"
    ][2]["clusters"]
    .copy()
)

final_subtype.name = "Subtype_raw"

print(final_subtype.value_counts().sort_index())

In [ ]:
# ==========================================================
# Biological subtype naming
# ==========================================================

final_subtype_named = final_subtype.map({
    1: "Metabolic",
    2: "Proliferative"
})

final_subtype_named.name = "Molecular_Subtype"

print(final_subtype_named.value_counts())

In [ ]:
Final_Subtype_df = pd.concat(
    [
        final_subtype,
        final_subtype_named
    ],
    axis=1
)

Final_Subtype_df.to_csv(
    os.path.join(
        results_dir,
        "LIHC_Final_Molecular_Subtype_D_Top30_K2.csv"
    )
)

In [ ]:
# ==========================================================
# Step13.27.1
# Final subtype Kaplan-Meier plot
# ==========================================================

import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

final_survival_df = (
    Final_Subtype_df
    .join(
        risk_df[
            ["OS_time", "OS", "RiskScore"]
        ],
        how="inner"
    )
)

g1 = final_survival_df[
    final_survival_df["Molecular_Subtype"]
    == "Metabolic"
]

g2 = final_survival_df[
    final_survival_df["Molecular_Subtype"]
    == "Proliferative"
]

lr = logrank_test(
    g1["OS_time"],
    g2["OS_time"],
    event_observed_A=g1["OS"],
    event_observed_B=g2["OS"]
)

print("Log-rank P =", lr.p_value)

In [ ]:
fig, ax = plt.subplots(
    figsize=(6.5, 5.5)
)

kmf1 = KaplanMeierFitter()
kmf2 = KaplanMeierFitter()

kmf1.fit(
    g1["OS_time"],
    event_observed=g1["OS"],
    label=f"Metabolic (n={len(g1)})"
)

kmf2.fit(
    g2["OS_time"],
    event_observed=g2["OS"],
    label=f"Proliferative (n={len(g2)})"
)

kmf1.plot_survival_function(
    ax=ax,
    ci_show=True
)

kmf2.plot_survival_function(
    ax=ax,
    ci_show=True
)

ax.set_xlabel("Overall survival time")
ax.set_ylabel("Overall survival probability")
ax.set_title("Overall Survival by Molecular Subtype")

ax.text(
    0.62,
    0.18,
    f"Log-rank P = {lr.p_value:.2e}",
    transform=ax.transAxes
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_KM_OS.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_KM_OS.svg"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ==========================================================
# Final subtype RiskScore comparison
# ==========================================================

import seaborn as sns
from scipy.stats import mannwhitneyu

plot_df = final_survival_df.copy()

g1_risk = plot_df.loc[
    plot_df["Molecular_Subtype"] == "Metabolic",
    "RiskScore"
]

g2_risk = plot_df.loc[
    plot_df["Molecular_Subtype"] == "Proliferative",
    "RiskScore"
]

U, p_risk = mannwhitneyu(
    g1_risk,
    g2_risk,
    alternative="two-sided"
)

print("RiskScore P =", p_risk)

In [ ]:
fig, ax = plt.subplots(
    figsize=(5.5, 5)
)

sns.boxplot(
    data=plot_df,
    x="Molecular_Subtype",
    y="RiskScore",
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="Molecular_Subtype",
    y="RiskScore",
    size=2,
    alpha=0.35,
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Risk score")
ax.set_title("Risk Score by Molecular Subtype")

ax.text(
    0.5,
    0.95,
    f"P = {p_risk:.2e}",
    ha="center",
    va="top",
    transform=ax.transAxes
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_RiskScore.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_RiskScore.svg"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ==========================================================
# Final subtype module eigengene comparison
# ==========================================================

module_plot_df = (
    Final_Subtype_df
    .join(
        ME_matrix[
            [
                "ME_blue",
                "ME_red",
                "ME_brown"
            ]
        ],
        how="inner"
    )
)

module_long = (
    module_plot_df
    .reset_index()
    .melt(
        id_vars=[
            "index",
            "Molecular_Subtype"
        ],
        value_vars=[
            "ME_blue",
            "ME_red",
            "ME_brown"
        ],
        var_name="Module",
        value_name="Eigengene"
    )
)

module_long["Module"] = (
    module_long["Module"]
    .replace({
        "ME_blue": "Blue",
        "ME_red": "Red",
        "ME_brown": "Brown"
    })
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 5.5)
)

sns.boxplot(
    data=module_long,
    x="Module",
    y="Eigengene",
    hue="Molecular_Subtype",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Module eigengene")
ax.set_title("WGCNA Module Activity by Molecular Subtype")

fig.tight_layout()

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_Module_Activity.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        figures_dir,
        "Final_Subtype_Module_Activity.svg"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ==========================================================
# Step13.29.1
# Prepare multivariable Cox dataset
# ==========================================================

cox_subtype_df = (
    Final_Subtype_df
    .join(
        risk_df[
            ["OS_time", "OS"]
        ],
        how="inner"
    )
    .join(
        clinical_analysis_df[
            [
                "Age",
                "Gender_group",
                "Stage_group"
            ]
        ],
        how="left"
    )
)

cox_subtype_df.head()

In [ ]:
# Proliferative = 1, Metabolic = 0
cox_subtype_df["Subtype_binary"] = (
    cox_subtype_df["Molecular_Subtype"]
    .map({
        "Metabolic": 0,
        "Proliferative": 1
    })
)

# Male = 1, Female = 0
cox_subtype_df["Male"] = (
    cox_subtype_df["Gender_group"]
    .map({
        "Female": 0,
        "Male": 1
    })
)

# Stage dummy variables
stage_dummies = pd.get_dummies(
    cox_subtype_df["Stage_group"],
    prefix="Stage",
    drop_first=True,
    dtype=int
)

cox_model_df = pd.concat(
    [
        cox_subtype_df[
            [
                "OS_time",
                "OS",
                "Subtype_binary",
                "Age",
                "Male"
            ]
        ],
        stage_dummies
    ],
    axis=1
).dropna()

print(cox_model_df.shape)
print(cox_model_df.columns.tolist())

In [ ]:
# ==========================================================
# Step13.29.2
# Multivariable Cox regression
# ==========================================================

from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    cox_model_df,
    duration_col="OS_time",
    event_col="OS"
)

cox_summary = cph.summary[
    [
        "coef",
        "exp(coef)",
        "exp(coef) lower 95%",
        "exp(coef) upper 95%",
        "p"
    ]
].copy()

cox_summary.columns = [
    "Coef",
    "HR",
    "HR_lower95",
    "HR_upper95",
    "P_value"
]

cox_summary

In [ ]:
cox_summary.to_csv(
    os.path.join(
        results_dir,
        "Final_Subtype_Multivariable_Cox.csv"
    )
)